In [ ]:
#r "BoSSSpad.dll"
using System;
using System.Data;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using System.Diagnostics;
using Microsoft.AspNetCore.Html;
using System.Text.RegularExpressions;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Solution.LevelSetTools.PhasefieldLevelSet;

Init();

A static droplet with two equal fluids is placed in a closed box.  
This testcase demonstrates the discrepancy between the fluid solver equilibrium contact angle  
and the phasefield level set equilibrium contact angle.  
Only as the interface thickness tends towards zero the latter value aproaches the first one.  
Due to this discrepancy parasitic currents are excited.  

The material values are chosen to be  
PhasefieldContactLine : $\rho = 1000, \sigma = 1.0, M = 1, \Delta t = 0.001$  
The cahn number is varied.  
The experiment is repeated for different prescribed static contact angles.

In [ ]:
string proj = "PhasefieldContactLine";
BoSSSshell.WorkflowMgm.Init(proj);

In [252]:
var sessions = BoSSSshell.WorkflowMgm.Sessions;
sessions.ForEach(s => display(s));

In [253]:
sessions.Count()

In [254]:
// Für Vislt notwendig

//sessions.Skip(0).Take(1).ForEach(session=> session.Export().To(Path.GetFullPath(Path.Join(".",session.Name))).WithSupersampling(2).Do());
// session.Export().To(Path.GetFullPath(Path.Join(".",session.Name))).WithSupersampling(2).WithShadowFields().Do();
// session.Export().To(Path.GetFullPath(Path.Join(".",session.Name))).WithSupersampling(2).Do();

In [255]:
// Process.Start("explorer.exe", sessions.Third().GetSessionDirectory());

In [256]:
sessions = sessions.Where(s => s.SuccessfulTermination).ToArray();

In [ ]:
var cldata = sessions.ToList().ReadLogDataForMovingContactLine(new string[] { "#timestep", "time", "contact-pointX", "contact-pointY", "contact-VelocityX", "contact-VelocityY", "contact-angle" })[0];
var energydata = sessions.ToList().ReadLogData("EnergyLogValues", new string[] {"#timestep", "time", "kineticEnergy", "surfaceEnergy", "totalEnergy", "kineticDissipation"});

In [292]:
static string ModName(string name){
    name = name.Replace("_","-");
    string[] nn = name.Split('-');
    int i = (int)((Convert.ToDouble(nn[1]) - Math.PI/4)/(Math.PI/12));
    if(nn.Length == 7){
        //int j = (int)(Convert.ToDouble(nn.Last())/0.02);
        int j = 4-(int)(Convert.ToDouble(nn.Last())/0.02);
        return $@"$\cAngle_{i} \quad \transition_{j}$";
    } else {
        return $@"$\cAngle_{i} \quad \mathrm{{reference}}$";
    }   
}

static PlotFormat ModFormat(string name){
    PlotFormat pf = new PlotFormat();
    pf.DashType = DashTypes.Solid;
    pf.PointSize = 0.7;
    pf.LineWidth = 1;
    pf.Style = Styles.LinesPoints;
    
    PointTypes[] pt = {PointTypes.UpperTriangle, PointTypes.LowerTriangle, PointTypes.Box, PointTypes.Diamond};
    LineColors[] lc = {LineColors.Red, LineColors.Green, LineColors.Blue, LineColors.Black};
    for(int i = 0; i < 4; i++){
        if(name.Contains($"cAngle_{i}")){
            pf.LineColor = lc[i];
        }
        
        if(name.Contains($"transition_{i}")){
            pf.PointType = pt[i];
        }
    }
    
    return pf;    
}

static Plot2Ddata.XYvalues[] ModSort(Plot2Ddata.XYvalues[] datagroup){    
    int[] weights = new int[datagroup.Length];

    for(int i = 0; i < datagroup.Length; i++){
        var group = datagroup[i];
        string[] nn = group.Name.Replace("$", "").Split(new char[] {'_',' '});
        if(nn.Length == 5){
            weights[i] = Convert.ToInt32(nn.Last());
        } else{
            weights[i] = 10;
        }
    }
    
    Array.Sort(weights, datagroup);
    
    return datagroup;
}

In [ ]:
Dictionary<int, List<ISessionInfo>> AnglesSessions = new Dictionary<int, List<ISessionInfo>>();
Dictionary<int, List<Plot2Ddata>[]> AnglesPlots = new Dictionary<int, List<Plot2Ddata>[]>();
double[] thetaS = new double[] {45.0, 60.0, 75.0, 90.0};

foreach(double theta in thetaS){
    int idx = (int)theta;
    List<ISessionInfo> sess = sessions.Where(s => s.Name.Contains((theta/180.0*Math.PI).ToString("N4"))).ToList();
    AnglesSessions[idx] = sess;
    
    List<Plot2Ddata>[] plts = new List<Plot2Ddata>[4];
    plts[0] = sess.ReadLogDataForMovingContactLine(new string[] { "#timestep", "time", "contact-pointX", "contact-pointY", "contact-VelocityX", "contact-VelocityY", "contact-angle" })[0];
    plts[1] = sess.ReadLogData("EnergyLogValues", new string[] {"#timestep", "time", "kineticEnergy", "surfaceEnergy", "totalEnergy", "kineticDissipation"});
    AnglesPlots[idx] = plts;
    
    plts[0].ElementAt(4).dataGroups.ForEach(xy => xy.Values = xy.Values.Select(y => Math.Abs(y - theta)).ToArray());
    plts[0].ElementAt(0).dataGroups.ForEach(xy => xy.Values = xy.Values.Select(y => Math.Abs(y + 0.5 * Math.Sin((theta/180.0*Math.PI)))).ToArray());

    //// subsample
    //plts[0].ForEach(plt => plt.dataGroups.ForEach(xy => {
    //    xy.Abscissas = xy.Abscissas.Where((x,i) => i % 25 == 0).ToArray();
    //    xy.Values = xy.Values.Where((x,i) => i % 25 == 0).ToArray();
    //}));
    //plts[1].ForEach(plt => plt.dataGroups.ForEach(xy => {
    //    xy.Abscissas = xy.Abscissas.Where((x,i) => i % 25 == 0).ToArray();
    //    xy.Values = xy.Values.Where((x,i) => i % 25 == 0).ToArray();
    //}));
    
    // rename
    plts[0].ForEach(plt => plt.dataGroups.ForEach(xy => xy.Name = ModName(xy.Name)));
    plts[1].ForEach(plt => plt.dataGroups.ForEach(xy => xy.Name = ModName(xy.Name)));
    
    // format
    plts[0].ForEach(plt => plt.dataGroups.ForEach(xy => xy.Format = ModFormat(xy.Name)));
    plts[1].ForEach(plt => plt.dataGroups.ForEach(xy => xy.Format = ModFormat(xy.Name)));
    
    plts[0].ForEach(plt => plt.dataGroups.ForEach(xy => xy.Format.Style = Styles.LinesPoints));
    plts[1].ForEach(plt => plt.dataGroups.ForEach(xy => xy.Format.Style = Styles.LinesPoints));
    
    // sort
    plts[0].ForEach(plt => plt.dataGroups = ModSort(plt.dataGroups));
    plts[1].ForEach(plt => plt.dataGroups = ModSort(plt.dataGroups));
    
    plts[2] = plts[0].Select(plt => new Plot2Ddata().Merge(plt)).ToList();
    plts[3] = plts[1].Select(plt => new Plot2Ddata().Merge(plt)).ToList();
    
    // subsample
    plts[2].ForEach(plt => plt.dataGroups.ForEach(xy => {
        xy.Abscissas = xy.Abscissas.Where((x,i) => i % 25 == 0).ToArray();
        xy.Values = xy.Values.Where((x,i) => i % 25 == 0).ToArray();
    }));
    plts[3].ForEach(plt => plt.dataGroups.ForEach(xy => {
        xy.Abscissas = xy.Abscissas.Where((x,i) => i % 25 == 0).ToArray();
        xy.Values = xy.Values.Where((x,i) => i % 25 == 0).ToArray();
    }));
    
    plts[2].ForEach(plt => plt.dataGroups.ForEach(xy => xy.Format.Style = Styles.Points));
    plts[3].ForEach(plt => plt.dataGroups.ForEach(xy => xy.Format.Style = Styles.Points));

}

In [314]:
Plot2Ddata[] plot = new Plot2Ddata[3];
plot[0] = new Plot2Ddata();
plot[1] = new Plot2Ddata();
plot[2] = new Plot2Ddata();

plot[0].Title = "Deviation in Contact Angle";
plot[1].Title = "Deviation in Contact Position";
plot[2].Title = "Kinetic Energy";

foreach(var kvp in AnglesPlots){
    plot[0] = plot[0].Merge(kvp.Value[0].ElementAt(4));
    plot[1] = plot[1].Merge(kvp.Value[0].ElementAt(0));
    plot[2] = plot[2].Merge(kvp.Value[1].ElementAt(0));
    
    //plot[0] = plot[0].Merge(kvp.Value[2].ElementAt(4));
    //plot[1] = plot[1].Merge(kvp.Value[2].ElementAt(0));
    //plot[2] = plot[2].Merge(kvp.Value[3].ElementAt(0));
}

Display preview of the relevant measurements.

In [306]:
//plot[0].YrangeMax = 5;
//plot[1].YrangeMax = 0.05;
//plot[2].YrangeMax = 1e-6;
//plot[0].YrangeMin = 0.0;
//plot[1].YrangeMin = 0.0;
//plot[2].YrangeMin = 0.0;

plot[0].PlotNow().Display();
plot[1].PlotNow().Display();
plot[2].PlotNow().Display();

In [307]:
//sessions.Skip(16).Take(1).ForEach(session=> session.Export().To(Path.GetFullPath(Path.Join(".",session.ProjectName,session.Name))).WithSupersampling(2).Do());


In [308]:
//plot[0].SaveToTextFile("PhasefieldComparisonContactAngle.txt");
//plot[1].SaveToTextFile("PhasefieldComparisonContactPoint.txt");
//plot[2].SaveToTextFile("PhasefieldComparisonKineticEnergy.txt");
//
//plot[0].SavePgfplotsFile("PhasefieldComparisonContactAngle.tex");
//plot[1].SavePgfplotsFile("PhasefieldComparisonContactPoint.tex");
//plot[2].SavePgfplotsFile("PhasefieldComparisonKineticEnergy.tex");

In [315]:
static void SavePgfplotsFileCustom(this Plot2Ddata plt, string filepath){

    Dictionary<string, string> Color = new Dictionary<string, string>()
    {
        {@"\cAngle_0", "red"},
        {@"\cAngle_1", "green"},
        {@"\cAngle_2", "blue"},
        {@"\cAngle_3", "black"}
    };
    
    Dictionary<string, string> Point = new Dictionary<string, string>()
    {
        {@"\transition_0", @"mark=triangle"},
        {@"\transition_1", @"mark=triangle, every mark/.append style={rotate=180}"},
        {@"\transition_2", @"mark=square"},
        {@"\transition_3", @"mark=square, every mark/.append style={rotate=45}"},
        {@"\mathrm{reference}", @"mark=."},
    };
    
    using (StreamWriter s = new StreamWriter(filepath)) {
        s.WriteLine(@"\begin{tikzpicture}");
        s.WriteLine(@"\begin{axis}");
        s.WriteLine(@"[legend style={at={(1.1,0.95)},anchor=north west,fill=none,draw=none,nodes=right}]");
        foreach (var group in plt.dataGroups) {
            string[] nn = group.Name.Replace("$","").Split(' ');
            //Write raw data points for the line
            s.WriteLine(@"\addplot [mark=none, solid, " + Color[nn[0]] +", forget plot] table{");
            for (int i = 0; i < group.Abscissas.Length; i++) {
                s.Write(group.Abscissas[i].ToString("E16", System.Globalization.NumberFormatInfo.InvariantInfo));
                s.Write("\t");
                s.Write(group.Values[i].ToString("E16", System.Globalization.NumberFormatInfo.InvariantInfo));
                s.Write("\n");
            }
            s.WriteLine(@"};");
            //Write every 25-th data points for the line
            s.WriteLine(@"\addplot [only marks, " + Color[nn[0]] +", " + Point[nn[2]] +"] table{");
            for (int i = 0; i < group.Abscissas.Length; i++) {
                if(i%25!=0)
                    continue;

                s.Write(group.Abscissas[i].ToString("E16", System.Globalization.NumberFormatInfo.InvariantInfo));
                s.Write("\t");
                s.Write(group.Values[i].ToString("E16", System.Globalization.NumberFormatInfo.InvariantInfo));
                s.Write("\n");
            }
            s.WriteLine(@"};");
            //s.WriteLine("\\addlegendentry{{{0}}};", group.Name);
        }
        s.WriteLine(@"\end{axis}");
        s.WriteLine(@"\end{tikzpicture}");
        s.Close(); 
    }
}

Create template for the use in the thesis

In [ ]:
//plot[0].SavePgfplotsFileCustom("PhasefieldComparisonContactAngle.tex");
//plot[1].SavePgfplotsFileCustom("PhasefieldComparisonContactPoint.tex");
//plot[2].SavePgfplotsFileCustom("PhasefieldComparisonKineticEnergy.tex");